<div style="background-color: #f0f4ff; padding: 16px 20px; border-left: 5px solid #4a6fa5; border-radius: 4px;">
<h2 style="margin: 0 0 8px 0;">Lab 1: Text Generation</h2>
<p style="margin: 0 0 16px 0; font-size: 13px; color: #555;">Adapted from Chan, Yee Kit, Erica Lai, and Yu Chen. "Teaching Tip: Teaching Undergraduate IS Students Hands-on Generative AI Development Skills." <em>Journal of Information Systems Education</em> 37(1), 2026.</p>
<hr style="border: none; border-top: 1px solid #ccc; margin-bottom: 16px;">
<b>First time in a coding environment?</b> You do not need to write any code — just run it and read the output. Click the play button or press <b>Shift + Enter</b>. Go top to bottom. If something breaks, go to <b>Runtime → Restart and run all</b>.<br><br>
<b>Table of contents</b> — Click the list icon on the left sidebar to jump between sections.<br>
<b>Two cell types</b> — Code cells have a play button. Text cells (like this one) have instructions.<br>
<b>Run order matters</b> — Skipping a cell may cause errors in later ones.
</div>

<div>
<h3>What is Text Generation?</h3>

<p>A 311 operator receives dozens of resident emails every day. Each response is written from scratch — no drafts, no templates, no assistance. The queue never gets shorter.</p>

<p>Text Generation means the AI reads a prompt and writes back. That single exchange is the foundation of every lab that follows. Every civic tool starts here.</p>

<p><strong>Labs 1–3 progression:</strong></p>
<table style="border-collapse:collapse;">
  <tbody>
    <tr style="background-color:#f0f4ff; font-weight:bold;"><td style="border:1px solid #ccc; padding:8px 16px;">Lab 1 ← you are here</td><td style="border:1px solid #ccc; padding:8px 16px;">AI responds to prompts — and you control how</td></tr>
    <tr><td style="border:1px solid #ccc; padding:8px 16px;">Lab 2</td><td style="border:1px solid #ccc; padding:8px 16px;">AI extracts structured data from messy text</td></tr>
    <tr style="background-color:#fafafa;"><td style="border:1px solid #ccc; padding:8px 16px;">Lab 3</td><td style="border:1px solid #ccc; padding:8px 16px;">AI reads images and generates civic reports</td></tr>
  </tbody>
</table>
</div>

# Setting Up Your Development Environment
- Install the Google Generative AI Python library.

In [1]:
!pip install -q google-generativeai

# Importing and Initializing Gemini

Store your API key in Colab Secrets:
 1. Click the key icon in the left sidebar
 2. Add a secret named `GEMINI_API_KEY`
 3. Paste your API key as the value
 4. Toggle "Notebook access" to ON

Get your free API key at: https://aistudio.google.com

In [2]:
import google.generativeai as genai
from google.colab import userdata
import time

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Gemini initialized successfully.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini initialized successfully.


In [3]:
import re
from google.api_core.exceptions import TooManyRequests

# Part 1: Your First AI Response

The function below sends a prompt to Gemini and returns its response.

Two inputs control the output:
- **`prompt`** — what the user says (the message)
- **`system_instruction`** — the role and rules you give the AI before any conversation begins

Run the cell and enter any prompt you like.

In [4]:
def get_completion(prompt, system_instruction="You are a helpful assistant.", max_retries=3):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=system_instruction
    )

    retries = 0
    while retries < max_retries:
        try:
            response = m.generate_content(prompt)
            time.sleep(12)  # stays under free tier rate limit (5 requests/min)
            return response.text
        except TooManyRequests as e:
            retries += 1
            if retries == max_retries:
                raise e # Re-raise if max retries reached

            # Extract retry time from the error message
            match = re.search(r'Please retry in (\d+\.?\d*)s\.', str(e))
            wait_time = 60 # Default wait time if not found
            if match:
                wait_time = float(match.group(1))

            print(f"Quota exceeded. Retrying in {wait_time:.2f} seconds... (Attempt {retries}/{max_retries})")
            time.sleep(wait_time + 1) # Add a small buffer
        except Exception as e:
            # Handle other potential errors
            raise e

    # Should not reach here if max_retries is handled correctly
    return ""

prompt = input("Enter a prompt: ")
response = get_completion(prompt)
print("\n--- AI Response ---")
print(response)

Enter a prompt: There's a large pothole on the corner of 1st Street and Santa Clara Avenue. It's been there for two weeks and almost damaged my tire.

--- AI Response ---
Oh no, that sounds really frustrating and potentially dangerous! Thanks for reporting it.

The best way to get this fixed is to report it directly to your local city's Public Works department or city services. Given "Santa Clara Avenue," I'm going to assume you're referring to **Santa Clara, California**.

Here's how you can report it:

1.  **Online Portal (Recommended):**
    *   Most cities have an online service request system. For Santa Clara, you can use their "Access Santa Clara" portal.
    *   Go to: [https://www.santaclaraca.gov/our-city/departments-g-z/public-works/report-a-problem](https://www.santaclaraca.gov/our-city/departments-g-z/public-works/report-a-problem)
    *   Look for "Service Request" or "Report a Problem."

2.  **Mobile App:**
    *   Santa Clara also has the "Access Santa Clara" mobile app 

# Part 2: The System Prompt — Where Design Decisions Live

The system prompt is not just a technical setting. **It is a policy decision.**

It determines:
- Who the AI thinks it is talking to
- What tone it uses
- What language it responds in
- What it will and won't do

The cell below shows a realistic civic use case: drafting a 311 response to a resident complaint.

Run it and read the output.

In [5]:
# A realistic 311 civic assistant system prompt
civic_system_prompt = """
You are a helpful assistant for the San Jose 311 civic reporting system.
A resident has submitted a complaint. Write a brief, polite acknowledgment
that: (1) confirms the report was received, (2) names the issue they reported,
and (3) tells them the expected response timeframe (3–5 business days for
non-emergency issues). Keep the response under 80 words.
"""

resident_complaint = (
    "There's a huge pothole on Alum Rock Ave near the library. "
    "I almost blew a tire yesterday. This has been here for months!"
)

response = get_completion(resident_complaint, civic_system_prompt)

print("--- Resident complaint ---")
print(resident_complaint)
print("\n--- AI-drafted response ---")
print(response)

--- Resident complaint ---
There's a huge pothole on Alum Rock Ave near the library. I almost blew a tire yesterday. This has been here for months!

--- AI-drafted response ---
Thank you for reporting the pothole on Alum Rock Ave. Your report has been received and logged. We understand your concern and appreciate you bringing this to our attention. For non-emergency issues like this, you can expect a response or action within 3-5 business days.


# Part 3: Prompt Design as an Equity Decision

San Jose is one of the most linguistically diverse cities in the United States.
Over 40% of residents speak a language other than English at home.

A 311 system that only works in English doesn't serve the whole city.

**Your task:** Modify the system prompt below to make the assistant multilingual.
Then test it with the Spanish-language complaint provided.

The one-line change you need to add to the system prompt is already shown as a comment.
Uncomment it, run the cell, and observe what changes.

In [6]:
# TASK: Uncomment the multilingual instruction below and re-run this cell.
# Compare the output to Part 2.

multilingual_system_prompt = """
You are a helpful assistant for the San Jose 311 civic reporting system.
A resident has submitted a complaint. Write a brief, polite acknowledgment
that: (1) confirms the report was received, (2) names the issue they reported,
and (3) tells them the expected response timeframe (3–5 business days for
non-emergency issues). Keep the response under 80 words.

# Uncomment the line below to enable multilingual support:
Detect the language the resident wrote in and respond in that same language.
"""

# A complaint submitted in Spanish
spanish_complaint = (
    "Hay un bache enorme en la calle Alum Rock cerca de la biblioteca. "
    "Casi dañé mi carro ayer. ¡Esto lleva meses así!"
)

response_es = get_completion(spanish_complaint, multilingual_system_prompt)

print("--- Resident complaint (Spanish) ---")
print(spanish_complaint)
print("\n--- AI-drafted response ---")
print(response_es)

--- Resident complaint (Spanish) ---
Hay un bache enorme en la calle Alum Rock cerca de la biblioteca. Casi dañé mi carro ayer. ¡Esto lleva meses así!

--- AI-drafted response ---
¡Gracias por tu reporte! Hemos recibido tu queja sobre el bache en la calle Alum Rock, cerca de la biblioteca. Tu reporte será revisado en los próximos 3 a 5 días hábiles.


# Part 4: Test Additional Languages

San Jose's largest non-English speaking communities include Spanish, Vietnamese, and Cantonese speakers.

The cell below tests all three with the **multilingual system prompt enabled**.

Run it and observe: does the AI respond correctly in each language?

In [7]:
# Multilingual system prompt — with the language detection line active
multilingual_prompt_active = """
You are a helpful assistant for the San Jose 311 civic reporting system.
A resident has submitted a complaint. Write a brief, polite acknowledgment
that: (1) confirms the report was received, (2) names the issue they reported,
and (3) tells them the expected response timeframe (3–5 business days for
non-emergency issues). Keep the response under 80 words.
Detect the language the resident wrote in and respond in that same language.
"""

test_messages = {
    "Spanish": "Hay basura ilegal tirada detrás de mi edificio en la calle Oak. Por favor, que alguien venga a limpiar.",
    "Vietnamese": "Có rác thải bất hợp pháp bị vứt sau tòa nhà của tôi trên đường Oak. Xin hãy cử người đến dọn dẹp.",
    "Cantonese": "我大廈後面嘅橡樹街有人非法棄置垃圾。請派人嚟清理。"
}

for language, message in test_messages.items():
    print(f"=== {language} ===")
    print(f"Input:  {message}")
    result = get_completion(message, multilingual_prompt_active)
    print(f"Output: {result}")
    print()

=== Spanish ===
Input:  Hay basura ilegal tirada detrás de mi edificio en la calle Oak. Por favor, que alguien venga a limpiar.


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 991.11ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1013.35ms


Output: ¡Gracias por contactarnos! Hemos recibido su informe sobre **basura ilegal** en la calle Oak. Un equipo evaluará la situación y esperamos ofrecer una respuesta en un plazo de 3 a 5 días hábiles.

=== Vietnamese ===
Input:  Có rác thải bất hợp pháp bị vứt sau tòa nhà của tôi trên đường Oak. Xin hãy cử người đến dọn dẹp.
Output: Chào quý vị,

Chúng tôi đã nhận được báo cáo của quý vị về vấn đề rác thải bất hợp pháp sau tòa nhà của quý vị trên đường Oak. Chúng tôi sẽ chuyển thông tin này đến đội phụ trách. Quý vị có thể mong đợi nhận được phản hồi trong vòng 3-5 ngày làm việc.

=== Cantonese ===
Input:  我大廈後面嘅橡樹街有人非法棄置垃圾。請派人嚟清理。
Output: 您的報告已收到。我們已收到您關於橡樹街非法棄置垃圾的報告。預計在3-5個工作天內處理此非緊急問題。感謝您的報告。



# Part 5: Reflection

**Write your answers directly in this text cell. Double-click to edit it.**

---

**Question 1 — Prompt as policy:**
In Part 3, a single line in the system prompt changed who the tool could serve.
If San Jose's 311 system currently responds only in English, who is excluded?
What would it take — beyond changing the AI prompt — to make the system genuinely accessible?

*Your answer here:*
Using an English-only 311 system would exclude a large number of San Jose residents and, by extension, households that speak a language other than English. This shift would most notably exclude Spanish-, Vietnamese-, and Cantonese-speaking people. The by-product of implementing such a shift would exclude speakers of other languages from making any requests to the city, which in turn would lead to a decreased standard of living for many people alike. To make this system genuinely accessible, it would require intake forms translated into a multitude of languages, bilingual operators to schedule phone follow-ups, and outreach translated by native speakers to make people aware that a system exists.
---

**Question 2 — Where AI judgment fits in:**
The AI drafts a response, but a city employee could review it before it's sent.
For what types of complaints would you want human review? For what types might
fully automated responses be acceptable? Where is the line, and who should draw it?

*Your answer here:*
Personally, I think a human should review every type of complaint. However, I understand that it is wishful thinking. On the practical level, fully automated replies are acceptable for minor, low-stakes issues with clear guidelines and steps for mitigation, such as broken streetlamps or graffiti. I strongly believe human review is absolutely necessary whenever a complaint involves a person as opposed to an object. Human intervention should also be required when an AI has low confidence in a response and when input mixes languages. A line should be drawn by people who are accountable and held responsible for the outcomes, such as the 311 director, city attorneys, and community advocates who represent the broader opinions of the overarching population.
---

**Question 3 — System prompt as a design artifact:**
You have now written and modified system prompts. In your own words:
what is a system prompt, what does it control, and why does it matter
who writes it?

*Your answer here:*
A system prompt is a set of instructions given to an AI before the user begins communicating with it. A system prompt controls factors such as the AI's tonality, the topics acceptable for discussion, and the formatting of responses. A system prompt is paramount to the success of an AI because it turns a generic chatbot into a bot with a real and specific purpose. Additionally, it is incredibly important who writes it because whoever creates the system prompt is deciding how the AI behaves on the backend. When the provided instructions are biased or missing important safety precautions, they have the potential to result in AI giving harmful or misleading answers.